# Step 5: final simple perceptron
University of Aberdeen ? Intelligent Robotics Design Project

This submission trains a single linear softmax layer on grayscale camera images. The selected configuration is unweighted, with learning rate 0.01, 3200 full-batch epochs, L2 0.0001 and final seed 42. Leave-one-session-out validation keeps each traversal independent. Balanced weighting remains an experimental fallback in the training functions. This notebook does not implement Step 6 control.

Run all cells in order in Google Colab. The official ZIP must contain top-level `forward/`, `left/`, `right/` folders and 385 JPEGs (313/36/36). Each filename must start with its session identifier and `__`, for example `session_20261103_01__forward_000001.jpg`. Preserve the original image bytes when packaging.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATASET_ZIP = Path('/content/drive/MyDrive/robotics/dataset_step05_official.zip')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/robotics/step05_final')
WORK_OUTPUT = Path('/content/step05_final')
CONFIG = dict(epochs=3200, learning_rate=0.01, l2=1e-4, seed=42)
CLASS_WEIGHTING = 'none'  # Selected submission; balanced is available via evaluate_fold.


In [ ]:
import json
import zipfile
import re
import hashlib
import shutil
from collections import Counter
from dataclasses import dataclass
from typing import Iterable
import cv2
import numpy as np
import tensorflow as tf
LABELS = ('forward', 'left', 'right')
LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}
EXPECTED_SHAPE = (50, 50, 3)
EXPECTED_COUNTS = {'forward': 313, 'left': 36, 'right': 36}
EXPECTED_SESSIONS = ('session_20261103_01', 'session_20261103_02', 'session_20261103_03')


## Validated NumPy training and metrics
The following functions use the repository training algorithm: float32 parameters, seeded normal initialization, full-batch weighted cross-entropy gradients and L2 on the kernel only. No Keras fitting is used. CV retains the repository convention of seed 42 plus the one-based fold index; final all-image training uses seed 42 exactly.


In [ ]:
@dataclass(frozen=True)
class SessionData:
    name: str
    x: np.ndarray
    y: np.ndarray
    paths: tuple[str, ...]


def preprocess_image(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise RuntimeError(f"Unreadable JPEG: {path}")

    if tuple(image.shape) != EXPECTED_SHAPE:
        raise RuntimeError(
            f"Unexpected image shape for {path}: "
            f"{tuple(image.shape)} != {EXPECTED_SHAPE}"
        )

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return gray.astype(np.float32).reshape(-1) / 255.0


def load_session(dataset_root: Path, session_name: str) -> SessionData:
    session_root = dataset_root / session_name
    if not session_root.is_dir():
        raise FileNotFoundError(f"Session directory not found: {session_root}")

    vectors: list[np.ndarray] = []
    labels: list[int] = []
    paths: list[str] = []

    for label in LABELS:
        folder = session_root / label
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing label directory: {folder}")

        files = sorted(folder.glob("*.jpg"))
        if not files:
            raise RuntimeError(f"No JPEGs found for {session_name}/{label}")

        for path in files:
            vectors.append(preprocess_image(path))
            labels.append(LABEL_TO_INDEX[label])
            paths.append(str(path))

    return SessionData(
        name=session_name,
        x=np.stack(vectors).astype(np.float32),
        y=np.asarray(labels, dtype=np.int32),
        paths=tuple(paths),
    )


def combine_sessions(sessions: Iterable[SessionData]) -> tuple[np.ndarray, np.ndarray]:
    sessions = tuple(sessions)
    if not sessions:
        raise ValueError("At least one session is required")
    return (
        np.concatenate([session.x for session in sessions], axis=0),
        np.concatenate([session.y for session in sessions], axis=0),
    )


def summarize_counts(y: np.ndarray) -> dict[str, int]:
    counts = Counter(int(value) for value in y.tolist())
    return {
        label: int(counts.get(index, 0))
        for index, label in enumerate(LABELS)
    }


def balanced_class_weights(y: np.ndarray) -> np.ndarray:
    """Return sklearn-style balanced weights N/(K*n_c) for LABELS."""
    if y.ndim != 1 or len(y) == 0:
        raise ValueError("y must be a non-empty 1D array")

    class_count = len(LABELS)
    counts = np.bincount(y.astype(np.int64), minlength=class_count)
    if np.any(counts <= 0):
        raise ValueError("Every class must be present to compute balanced weights")

    weights = len(y) / (class_count * counts.astype(np.float64))
    return weights.astype(np.float32)


def _softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def train_perceptron(
    x: np.ndarray,
    y: np.ndarray,
    *,
    epochs: int,
    learning_rate: float,
    l2: float,
    seed: int,
    class_weights: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, list[float]]:
    if epochs <= 0:
        raise ValueError("epochs must be positive")
    if learning_rate <= 0:
        raise ValueError("learning_rate must be positive")
    if l2 < 0:
        raise ValueError("l2 must be non-negative")

    rng = np.random.default_rng(seed)
    feature_count = x.shape[1]
    class_count = len(LABELS)

    if class_weights is None:
        effective_class_weights = np.ones(class_count, dtype=np.float32)
    else:
        effective_class_weights = np.asarray(class_weights, dtype=np.float32)
        if effective_class_weights.shape != (class_count,):
            raise ValueError(
                f"class_weights must have shape {(class_count,)}, "
                f"got {effective_class_weights.shape}"
            )
        if not np.all(np.isfinite(effective_class_weights)):
            raise ValueError("class_weights must be finite")
        if np.any(effective_class_weights <= 0):
            raise ValueError("class_weights must be strictly positive")

    sample_weights = effective_class_weights[y]
    total_sample_weight = float(sample_weights.sum())
    if total_sample_weight <= 0:
        raise ValueError("Total sample weight must be positive")

    weights = rng.normal(
        loc=0.0,
        scale=0.01,
        size=(feature_count, class_count),
    ).astype(np.float32)
    bias = np.zeros(class_count, dtype=np.float32)

    targets = np.eye(class_count, dtype=np.float32)[y]
    losses: list[float] = []

    for _ in range(int(epochs)):
        logits = x @ weights + bias
        probabilities = _softmax(logits)

        clipped = np.clip(
            probabilities[np.arange(len(y)), y],
            1e-8,
            1.0,
        )
        data_loss = -float(
            np.sum(sample_weights * np.log(clipped)) / total_sample_weight
        )
        reg_loss = 0.5 * float(l2) * float(np.sum(weights * weights))
        losses.append(data_loss + reg_loss)

        gradient_logits = (
            (probabilities - targets)
            * sample_weights[:, None]
            / total_sample_weight
        )
        gradient_weights = x.T @ gradient_logits + float(l2) * weights
        gradient_bias = gradient_logits.sum(axis=0)

        weights -= float(learning_rate) * gradient_weights.astype(np.float32)
        bias -= float(learning_rate) * gradient_bias.astype(np.float32)

    return weights, bias, losses


def predict_scores(
    x: np.ndarray,
    weights: np.ndarray,
    bias: np.ndarray,
) -> np.ndarray:
    return _softmax(x @ weights + bias)


def predict_labels(
    x: np.ndarray,
    weights: np.ndarray,
    bias: np.ndarray,
) -> np.ndarray:
    return np.argmax(predict_scores(x, weights, bias), axis=1).astype(np.int32)


def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    matrix = np.zeros((len(LABELS), len(LABELS)), dtype=np.int64)
    for truth, prediction in zip(y_true, y_pred):
        matrix[int(truth), int(prediction)] += 1
    return matrix


def metrics_from_confusion(matrix: np.ndarray) -> dict:
    if matrix.shape != (len(LABELS), len(LABELS)):
        raise ValueError("Unexpected confusion matrix shape")

    total = int(matrix.sum())
    correct = int(np.trace(matrix))
    accuracy = correct / total if total else 0.0

    per_class = {}
    f1_values = []

    for index, label in enumerate(LABELS):
        tp = int(matrix[index, index])
        fn = int(matrix[index, :].sum() - tp)
        fp = int(matrix[:, index].sum() - tp)
        support = int(matrix[index, :].sum())

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2.0 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )
        f1_values.append(f1)

        per_class[label] = {
            "support": support,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    return {
        "accuracy": accuracy,
        "macro_f1": float(np.mean(f1_values)),
        "per_class": per_class,
        "confusion_matrix": matrix.tolist(),
    }


def print_metrics(title: str, metrics: dict) -> None:
    print()
    print(title)
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Macro F1 : {metrics['macro_f1']:.4f}")
    print("Confusion matrix (rows=true, cols=pred; forward,left,right):")
    for row in metrics["confusion_matrix"]:
        print(" ", row)

    print("Per-class:")
    for label in LABELS:
        item = metrics["per_class"][label]
        print(
            f"  {label:8s} "
            f"support={item['support']:3d} "
            f"precision={item['precision']:.4f} "
            f"recall={item['recall']:.4f} "
            f"f1={item['f1']:.4f}"
        )


def evaluate_fold(
    *,
    train_sessions: tuple[SessionData, ...],
    held_out: SessionData,
    epochs: int,
    learning_rate: float,
    l2: float,
    seed: int,
    class_weighting: str,
) -> tuple[dict, np.ndarray, np.ndarray]:
    x_train, y_train = combine_sessions(train_sessions)

    if class_weighting == "balanced":
        fold_class_weights = balanced_class_weights(y_train)
    elif class_weighting == "none":
        fold_class_weights = None
    else:
        raise ValueError(f"Unsupported class_weighting: {class_weighting}")

    weights, bias, losses = train_perceptron(
        x_train,
        y_train,
        epochs=epochs,
        learning_rate=learning_rate,
        l2=l2,
        seed=seed,
        class_weights=fold_class_weights,
    )

    predictions = predict_labels(held_out.x, weights, bias)
    matrix = confusion_matrix(held_out.y, predictions)
    metrics = metrics_from_confusion(matrix)

    metrics.update(
        {
            "held_out_session": held_out.name,
            "train_sessions": [session.name for session in train_sessions],
            "train_counts": summarize_counts(y_train),
            "evaluation_counts": summarize_counts(held_out.y),
            "class_weighting": class_weighting,
            "class_weights": (
                {
                    label: float(fold_class_weights[index])
                    for index, label in enumerate(LABELS)
                }
                if fold_class_weights is not None
                else {label: 1.0 for label in LABELS}
            ),
            "final_training_loss": float(losses[-1]),
        }
    )
    return metrics, held_out.y, predictions




## Check and load the official ZIP
Images are decoded directly from the archive without resizing. Session membership is recovered from filenames; counts and dimensions must match before training.


In [ ]:
def load_official_zip(path):
    grouped = {name: [] for name in EXPECTED_SESSIONS}
    counts = Counter()
    seen = set()
    with zipfile.ZipFile(path) as archive:
        for member in sorted(archive.infolist(), key=lambda item: item.filename):
            if member.is_dir():
                continue
            parts = member.filename.split('/')
            if len(parts) != 2 or parts[0] not in LABELS:
                raise ValueError(f'Expected top-level label/JPEG: {member.filename}')
            label, filename = parts
            if Path(filename).suffix.lower() not in ('.jpg', '.jpeg'):
                raise ValueError(f'Expected JPEG: {filename}')
            if member.filename in seen:
                raise ValueError(f'Duplicate archive member: {member.filename}')
            seen.add(member.filename)
            match = re.fullmatch(r'(session_\d{8}_\d{2})__(.+)', filename)
            if match is None or match[1] not in grouped:
                raise ValueError(f'Missing or unexpected session prefix: {filename}')
            image = cv2.imdecode(np.frombuffer(archive.read(member), dtype=np.uint8), cv2.IMREAD_COLOR)
            if image is None or image.shape != EXPECTED_SHAPE:
                raise ValueError(f'Expected 50x50x3 JPEG: {filename}')
            vector = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.float32).reshape(-1) / 255.0
            grouped[match[1]].append((LABEL_TO_INDEX[label], filename, vector))
            counts[label] += 1
    if dict(counts) != EXPECTED_COUNTS or sum(counts.values()) != 385:
        raise ValueError(f'Unexpected class counts: {dict(counts)}')
    sessions = []
    for name, records in grouped.items():
        # Match repository ordering: label order, then filename within label.
        records.sort(key=lambda row: (row[0], row[1]))
        if set(row[0] for row in records) != set(range(3)):
            raise ValueError(f'Each session must contain all classes: {name}')
        sessions.append(SessionData(name, np.stack([r[2] for r in records]),
                                    np.asarray([r[0] for r in records], dtype=np.int32),
                                    tuple(r[1] for r in records)))
    return tuple(sessions)

sessions = load_official_zip(DATASET_ZIP)
x_all, y_all = combine_sessions(sessions)
print('Verified:', x_all.shape, summarize_counts(y_all))
for session in sessions:
    print(session.name, summarize_counts(session.y))


In [ ]:
folds, truths, predictions = [], [], []
for index, held_out in enumerate(sessions, start=1):
    training = tuple(s for s in sessions if s.name != held_out.name)
    fold_config = {**CONFIG, 'seed': CONFIG['seed'] + index}
    metrics, truth, prediction = evaluate_fold(
        train_sessions=training, held_out=held_out,
        class_weighting=CLASS_WEIGHTING, **fold_config)
    metrics['seed'] = fold_config['seed']
    folds.append(metrics)
    truths.append(truth)
    predictions.append(prediction)
    print_metrics(f'Held out: {held_out.name}', metrics)
pooled = metrics_from_confusion(confusion_matrix(np.concatenate(truths), np.concatenate(predictions)))
print_metrics('Pooled leave-one-session-out validation', pooled)


## Final model and official HDF5 export
Train on all 385 images with the selected unweighted objective. Transfer the NumPy kernel and bias to one Keras Dense layer and verify both probabilities and class predictions before saving. Input preprocessing remains external to the exported model.


In [ ]:
if CLASS_WEIGHTING != 'none':
    raise ValueError('Official final export requires the selected unweighted configuration')
weights, bias, losses = train_perceptron(x_all, y_all, class_weights=None, **CONFIG)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(2500,), dtype='float32'),
    tf.keras.layers.Dense(3, activation='softmax', name='direction'),
])
model.get_layer('direction').set_weights([weights, bias])
numpy_scores = predict_scores(x_all, weights, bias)
keras_scores = model(x_all, training=False).numpy()
np.testing.assert_allclose(keras_scores, numpy_scores, rtol=1e-5, atol=1e-6)
np.testing.assert_array_equal(keras_scores.argmax(axis=1), numpy_scores.argmax(axis=1))
WORK_OUTPUT.mkdir(parents=True, exist_ok=True)
model_path = WORK_OUTPUT / 'maze_navigation_simple_perceptron_final.h5'
model.save(model_path)
reloaded = tf.keras.models.load_model(model_path, compile=False)
np.testing.assert_allclose(reloaded(x_all, training=False).numpy(), numpy_scores, rtol=1e-5, atol=1e-6)
metadata = {
    'model_file': model_path.name, 'architecture': '2500 -> Dense(3, softmax)',
    'labels': list(LABELS), 'config': {**CONFIG, 'class_weighting': CLASS_WEIGHTING},
    'preprocessing': 'OpenCV BGR2GRAY -> float32 /255 -> flatten 2500',
    'input_image_shape': list(EXPECTED_SHAPE), 'training_counts': summarize_counts(y_all),
    'training_total': len(y_all), 'dataset_zip_sha256': hashlib.sha256(DATASET_ZIP.read_bytes()).hexdigest(),
    'evaluation_protocol': 'leave_one_session_out', 'folds': folds, 'pooled': pooled,
    'final_training_loss': losses[-1],
    'prediction_max_absolute_error': float(np.max(np.abs(keras_scores - numpy_scores))),
    'versions': {'numpy': np.__version__, 'opencv': cv2.__version__, 'tensorflow': tf.__version__},
    'status': 'Step 5 offline submission; Step 6 not completed',
}
metadata_path = WORK_OUTPUT / 'maze_navigation_simple_perceptron_final.json'
metadata_path.write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
for artifact in (model_path, metadata_path):
    shutil.copy2(artifact, DRIVE_OUTPUT / artifact.name)
print('Verified .h5 and metrics saved to', DRIVE_OUTPUT)


## Submission
Submit this completed Colab `.ipynb` (save a copy with outputs) and `maze_navigation_simple_perceptron_final.h5`. Keep the JSON metrics alongside them as reproducibility evidence. Offline validation does not establish live maze-navigation performance; Step 6 remains separate.
